In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")

# Create processed directory if it does not already exist
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Load raw NAV history
nav = pd.read_csv(RAW_DATA_DIR / "02_nav_history.csv")

print("Shape:", nav.shape)
print("\nColumns:")
print(nav.columns.tolist())

print("\nData Types:")
print(nav.dtypes)

print("\nFirst 5 rows:")
display(nav.head())

Shape: (46000, 3)

Columns:
['amfi_code', 'date', 'nav']

Data Types:
amfi_code      int64
date             str
nav          float64
dtype: object

First 5 rows:


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [4]:
# NAV history quality checks before cleaning

print("Missing values:")
print(nav.isnull().sum())

print("\nExact duplicate rows:", nav.duplicated().sum())

print(
    "Duplicate AMFI code + date combinations:",
    nav.duplicated(subset=["amfi_code", "date"]).sum()
)

print("\nNAV <= 0:", (nav["nav"] <= 0).sum())

print("\nNAV statistics:")
display(nav["nav"].describe())

print("\nUnique schemes:", nav["amfi_code"].nunique())

print("\nDate range:")
print("Minimum date:", nav["date"].min())
print("Maximum date:", nav["date"].max())

Missing values:
amfi_code    0
date         0
nav          0
dtype: int64

Exact duplicate rows: 0
Duplicate AMFI code + date combinations: 0

NAV <= 0: 0

NAV statistics:


count    46000.000000
mean       269.570265
std        577.187060
min         26.136600
25%         69.170425
50%        122.732150
75%        260.338675
max       4268.549700
Name: nav, dtype: float64


Unique schemes: 40

Date range:
Minimum date: 2022-01-03
Maximum date: 2026-05-29


In [5]:
# Convert date column to datetime
nav["date"] = pd.to_datetime(nav["date"])

# Sort by AMFI code and date
nav = nav.sort_values(["amfi_code", "date"]).reset_index(drop=True)

print("Data Types:")
print(nav.dtypes)

print("\nFirst 5 rows:")
display(nav.head())

print("\nLast 5 rows:")
display(nav.tail())

Data Types:
amfi_code             int64
date         datetime64[us]
nav                 float64
dtype: object

First 5 rows:


,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639



Last 5 rows:


,amfi_code,date,nav
45995,149324,2026-05-25,292.4810
45996,149324,2026-05-26,291.2707
45997,149324,2026-05-27,288.8007
45998,149324,2026-05-28,280.6873
45999,149324,2026-05-29,279.7511


In [6]:
# Check number of records for each AMFI scheme

scheme_counts = (
    nav.groupby("amfi_code")
       .size()
       .reset_index(name="records")
)

print("Total Schemes:", len(scheme_counts))

display(scheme_counts.head(10))

print("\nRecord count summary:")
display(scheme_counts["records"].describe())

Total Schemes: 40


,amfi_code,records
0,100016,1150
1,100025,1150
2,100033,1150
3,101206,1150
4,101207,1150
5,101208,1150
6,102885,1150
7,102886,1150
8,102887,1150
9,118632,1150



Record count summary:


count      40.0
mean     1150.0
std         0.0
min      1150.0
25%      1150.0
50%      1150.0
75%      1150.0
max      1150.0
Name: records, dtype: float64

In [7]:
# Save cleaned NAV history

output_file = PROCESSED_DATA_DIR / "02_nav_history_cleaned.csv"

nav.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Rows:", len(nav))
print("Columns:", len(nav.columns))

Saved: ..\data\processed\02_nav_history_cleaned.csv
Rows: 46000
Columns: 3


In [8]:
# Load investor transactions dataset

transactions = pd.read_csv(RAW_DATA_DIR / "08_investor_transactions.csv")

print("Shape:", transactions.shape)

print("\nColumns:")
print(transactions.columns.tolist())

print("\nData Types:")
print(transactions.dtypes)

print("\nFirst 5 rows:")
display(transactions.head())

Shape: (32778, 13)

Columns:
['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']

Data Types:
investor_id               str
transaction_date          str
amfi_code               int64
transaction_type          str
amount_inr              int64
state                     str
city                      str
city_tier                 str
age_group                 str
gender                    str
annual_income_lakh    float64
payment_mode              str
kyc_status                str
dtype: object

First 5 rows:


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [9]:
# -----------------------------
# Investor Transactions Quality Check
# -----------------------------

print("Missing Values:")
print(transactions.isnull().sum())

print("\nExact Duplicate Rows:")
print(transactions.duplicated().sum())

print("\nAmount <= 0:")
print((transactions["amount_inr"] <= 0).sum())

print("\nTransaction Types:")
print(transactions["transaction_type"].value_counts())

print("\nKYC Status:")
print(transactions["kyc_status"].value_counts())

print("\nPayment Modes:")
print(transactions["payment_mode"].value_counts())

print("\nDate Range:")
print("Minimum:", transactions["transaction_date"].min())
print("Maximum:", transactions["transaction_date"].max())

Missing Values:
investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64

Exact Duplicate Rows:
0

Amount <= 0:
0

Transaction Types:
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

KYC Status:
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

Payment Modes:
payment_mode
Net Banking    8250
Cheque         8228
UPI            8154
Mandate        8146
Name: count, dtype: int64

Date Range:
Minimum: 2024-01-01
Maximum: 2025-05-30


In [10]:
# -----------------------------
# Clean Investor Transactions
# -----------------------------

# Convert transaction_date to datetime
transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"]
)

# Standardize transaction_type
transactions["transaction_type"] = (
    transactions["transaction_type"]
    .str.strip()
    .str.title()
)

# Standardize KYC status
transactions["kyc_status"] = (
    transactions["kyc_status"]
    .str.strip()
    .str.title()
)

# Standardize payment mode
transactions["payment_mode"] = (
    transactions["payment_mode"]
    .str.strip()
)

# Final validation
print("Transaction Types:")
print(transactions["transaction_type"].value_counts())

print("\nKYC Status:")
print(transactions["kyc_status"].value_counts())

print("\nData Types:")
print(transactions.dtypes)

# Save cleaned dataset
output_file = PROCESSED_DATA_DIR / "08_investor_transactions_cleaned.csv"

transactions.to_csv(output_file, index=False)

print("\nSaved:", output_file)
print("Rows:", len(transactions))
print("Columns:", len(transactions.columns))

Transaction Types:
transaction_type
Sip           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

KYC Status:
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

Data Types:
investor_id                      str
transaction_date      datetime64[us]
amfi_code                      int64
transaction_type                 str
amount_inr                     int64
state                            str
city                             str
city_tier                        str
age_group                        str
gender                           str
annual_income_lakh           float64
payment_mode                     str
kyc_status                       str
dtype: object

Saved: ..\data\processed\08_investor_transactions_cleaned.csv
Rows: 32778
Columns: 13


In [11]:
# Standardize transaction type exactly as required

transactions["transaction_type"] = (
    transactions["transaction_type"]
    .str.strip()
    .replace({
        "Sip": "SIP",
        "sip": "SIP",
        "SIP": "SIP",
        "Lumpsum": "Lumpsum",
        "Redemption": "Redemption"
    })
)

print(transactions["transaction_type"].value_counts())

# Save again
transactions.to_csv(
    PROCESSED_DATA_DIR / "08_investor_transactions_cleaned.csv",
    index=False
)

transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64


In [13]:
# Load Scheme Performance dataset

performance = pd.read_csv(RAW_DATA_DIR / "07_scheme_performance.csv")

print("Shape:", performance.shape)

print("\nColumns:")
print(performance.columns.tolist())

print("\nData Types:")
print(performance.dtypes)

print("\nFirst 5 rows:")
display(performance.head())

Shape: (40, 19)

Columns:
['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']

Data Types:
amfi_code               int64
scheme_name               str
fund_house                str
category                  str
plan                      str
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade                str
dtype: object

First 5 rows:


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [14]:
# -----------------------------
# Scheme Performance Quality Check
# -----------------------------

print("Missing Values:")
print(performance.isnull().sum())

print("\nExact Duplicate Rows:")
print(performance.duplicated().sum())

print("\nDuplicate AMFI Codes:")
print(performance.duplicated(subset=["amfi_code"]).sum())

print("\nMorningstar Ratings:")
print(performance["morningstar_rating"].value_counts().sort_index())

print("\nRisk Grades:")
print(performance["risk_grade"].value_counts())

print("\nNumeric Summary:")
display(performance.describe())

Missing Values:
amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
dtype: int64

Exact Duplicate Rows:
0

Duplicate AMFI Codes:
0

Morningstar Ratings:
morningstar_rating
3     7
4    16
5    17
Name: count, dtype: int64

Risk Grades:
risk_grade
Moderate           16
High                8
Very High           6
Low                 6
Moderately High     4
Name: count, dtype: int64

Numeric Summary:


,amfi_code,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating
count,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.00000,40.000000,40.000000
mean,120247.000000,14.376000,14.089000,14.516750,12.835500,1.253500,0.873250,1.361750,2.082500,14.962500,-19.200250,26091.60000,1.237000,4.250000
std,14534.998667,4.883023,4.617253,4.454021,4.740972,0.447412,0.224846,1.475805,2.203144,6.669282,8.819164,13809.11134,0.386584,0.742484
min,100016.000000,4.260000,5.140000,5.430000,3.960000,0.510000,0.220000,0.800000,1.030000,0.500000,-33.500000,979.00000,0.550000,3.000000
25%,118632.750000,11.735000,12.035000,12.340000,10.690000,0.887500,0.890000,0.865000,1.270000,14.000000,-25.062500,17400.50000,0.787500,4.000000
50%,119551.500000,14.620000,14.205000,14.185000,13.090000,1.205000,0.960000,0.925000,1.445000,14.000000,-20.600000,26713.00000,1.425000,4.000000
75%,120842.250000,16.392500,15.882500,17.585000,14.775000,1.700000,1.000000,0.985000,1.637500,19.000000,-14.255000,38125.00000,1.540000,5.000000
max,149324.000000,24.930000,23.390000,23.800000,22.160000,1.980000,1.040000,7.680000,10.370000,25.000000,-2.230000,49046.00000,1.640000,5.000000


In [15]:
# -----------------------------
# Clean Scheme Performance
# -----------------------------

# Remove extra whitespace from text columns
text_columns = performance.select_dtypes(include="object").columns

for col in text_columns:
    performance[col] = performance[col].str.strip()

# Remove duplicate rows if any
performance = performance.drop_duplicates()

# Save cleaned dataset
output_file = PROCESSED_DATA_DIR / "07_scheme_performance_cleaned.csv"

performance.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Rows:", len(performance))
print("Columns:", len(performance.columns))

Saved: ..\data\processed\07_scheme_performance_cleaned.csv
Rows: 40
Columns: 19


C:\Users\Dell\AppData\Local\Temp\ipykernel_18484\1879824696.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = performance.select_dtypes(include="object").columns
